# Контрольная работа 2



Сделайте копию ноутбука

Присвойте переменной `v` значение свой табельный номер ИСУ, это нужно для определения вашего варианта. Запустите код в следующей ячейке.

In [ ]:
from datetime import datetime
current_datetime = datetime.now()
print(current_datetime)
v = 409326

2026-05-25 09:17:52.669184


# Общее задание

Нужно решить задачу бинарной классификации, предварительно построив признаковое описание объектов на основе нескольких таблиц.

Целевая переменная - пол клиента.

В качестве модели нужно использовать нейронную сеть, которую нужно строить с помощью `keras` или `torch` на выбор студента.

# Данные

Для последующих заданий будем использовать обезличенные транзакционные банковские данные. Для этого считайте в переменные **transactions_short, tr_mcc_codes и gender_train** из одноимённых таблиц из папки data.

###  Описание данных
#### Таблица ```transactions_short.csv```
##### Описание
Таблица содержит историю транзакций клиентов банка за один год и три месяца.

##### Формат данных

```
customer_id,tr_datetime,mcc_code,tr_type,amount,term_id
111111,15 01:40:52,1111,1000,-5224,111111
111112,15 15:18:32,3333,2000,-100,11122233
...
```
##### Описание полей

 - ```customer_id``` — идентификатор клиента;
 - ```tr_datetime``` — день и время совершения транзакции (дни нумеруются с начала данных);
 - ```mcc_code``` — mcc-код транзакции;
 - ```tr_type``` — тип транзакции;
 - ```amount``` — сумма транзакции в условных единицах со знаком; ```+``` — начисление средств клиенту (приходная транзакция), ```-``` — списание средств (расходная транзакция);
 - ```term_id``` — идентификатор терминала;


#### Таблица ```gender_train.csv```

##### Описание
Данная таблица содержит информацию по полу для части клиентов, для которых он известен. Для остальных клиентов пол неизвестен.

##### Формат данных
```
customer_id,gender
111111,0
111112,1
...
```

##### Описание полей
 - ```customer_id``` — идентификатор клиента;
 - ```gender``` — пол клиента;

### Таблица ```tr_mcc_codes.csv```

##### Описание
Данная таблица содержит описание mcc-кодов транзакций.

##### Формат данных
```
mcc_code;mcc_description
1000;словесное описание mcc-кода 1000
2000;словесное описание mcc-кода 2000
...
```

##### Описание полей
 - ```mcc_code``` – mcc-код транзакции;
 - ```mcc_description``` — описание mcc-кода транзакции.


In [ ]:
!gdown 1FG1fopcmvMZ7GBaBOqQipccSeFoMUvNT

Downloading...
From: https://drive.google.com/uc?id=1FG1fopcmvMZ7GBaBOqQipccSeFoMUvNT
To: /content/gender_train.csv
100% 99.9k/99.9k [00:00<00:00, 24.5MB/s]


In [ ]:
!gdown 10J8RzMIhoYHiad49r-oWNMAk-V5lo3OE

Downloading...
From: https://drive.google.com/uc?id=10J8RzMIhoYHiad49r-oWNMAk-V5lo3OE
To: /content/tr_mcc_codes.csv
100% 14.9k/14.9k [00:00<00:00, 26.8MB/s]


In [ ]:
!gdown 1viqn9Y3kX2JjWyWAFupTouIwyIPxw1Ys

Downloading...
From: https://drive.google.com/uc?id=1viqn9Y3kX2JjWyWAFupTouIwyIPxw1Ys
To: /content/transactions_short.csv
100% 104M/104M [00:01<00:00, 58.6MB/s] 


Либо скачайте данные отсюда: https://drive.google.com/drive/folders/1YAMe7MiTxA-RSSd8Ex2p-L0Dspe6Gs4L?usp=sharing Для задания нужны файлы: gender_train.csv, tr_mcc_codes.csv и transactions_short.csv (это большой файл, около 100M)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              precision_score, recall_score, classification_report)
import matplotlib.pyplot as plt

transactions_short = pd.read_csv('transactions_short.csv')
tr_mcc_codes       = pd.read_csv('tr_mcc_codes.csv', sep=';')
gender_train       = pd.read_csv('gender_train.csv')

print('transactions_short:', transactions_short.shape)
print('tr_mcc_codes:',       tr_mcc_codes.shape)
print('gender_train:',       gender_train.shape)
print(transactions_short.head(3))


# Задание 1



В задании требуется на основе нескольких таблиц с данными сделать признаковое описание объектов.

Объектами являются клиенты. Клиенты идентифицируются с помощью `customer_id`, которые есть в таблицах **transactions_short** и **gender_train**. В качестве призаков нужно использовать даннее по категориям транзаций: ```mcc_code``` — mcc-код транзакции есть в таблицах **transactions_short** и **tr_mcc_codes**


Сформируйте вектора для описания клиентов. Опишите логику построения этих векторов.


Сделайте нормализацию значений признаков так, чтобы все означения менялись от 0
до 1.

In [ ]:
# ── Построение признакового описания клиентов ─────────────────────────────
#
# Логика:
#   Объект = клиент (customer_id).
#   Признаки считаем ОТДЕЛЬНО для приходных и расходных транзакций,
#   так как они несут разную информацию о поведении клиента.
#   Для каждого mcc_code:
#     - сумма расходных транзакций (amount < 0)  → колонки с суффиксом _out
#     - сумма приходных транзакций (amount > 0)  → колонки с суффиксом _in
#   Нормализация MinMaxScaler приводит все значения к диапазону [0, 1].

incoming = transactions_short[transactions_short['amount'] > 0].copy()
outgoing = transactions_short[transactions_short['amount'] < 0].copy()
outgoing['amount'] = outgoing['amount'].abs()

# Pivot для приходных транзакций
pivot_in = incoming.pivot_table(
    index='customer_id', columns='mcc_code',
    values='amount', aggfunc='sum', fill_value=0
)
pivot_in.columns  = [f'{c}_in'  for c in pivot_in.columns]

# Pivot для расходных транзакций
pivot_out = outgoing.pivot_table(
    index='customer_id', columns='mcc_code',
    values='amount', aggfunc='sum', fill_value=0
)
pivot_out.columns = [f'{c}_out' for c in pivot_out.columns]

# Объединяем оба блока признаков
feature_df = pivot_in.join(pivot_out, how='outer').fillna(0)

# Нормализация MinMaxScaler: все значения в [0, 1]
scaler = MinMaxScaler()
feature_scaled = scaler.fit_transform(feature_df)
feature_df_norm = pd.DataFrame(
    feature_scaled, index=feature_df.index, columns=feature_df.columns
)

print(f"Матрица признаков: {feature_df_norm.shape[0]} клиентов × {feature_df_norm.shape[1]} признаков")
print(f"  (приходные: {len(pivot_in.columns)}, расходные: {len(pivot_out.columns)})")
print(f"Разреженность: {(feature_df_norm == 0).values.mean():.1%} нулевых значений")

# ── Q1.2: Количество непустых признаков на клиента ───────────────────────
nonzero_counts = (feature_df_norm > 0).sum(axis=1).rename('nonzero_features')
sorted_clients = nonzero_counts.sort_values(ascending=False).reset_index()
sorted_clients.columns = ['customer_id', 'nonzero_features']

print("\n── Q1.2: Клиенты, отсортированные по количеству непустых признаков ──")
print(sorted_clients.to_string(index=False))

# ── Q1.3: Топ-10 с наибольшим числом непустых признаков ──────────────────
print("\n── Q1.3: Топ-10 клиентов с наибольшим числом непустых признаков ──")
print(sorted_clients.head(10).to_string(index=False))

# ── Q1.4: Топ-10 с наименьшим числом непустых признаков ──────────────────
print("\n── Q1.4: Топ-10 клиентов с наименьшим числом непустых признаков ──")
print(sorted_clients.tail(10).to_string(index=False))


In [ ]:
# ── Q1.1: Для каждой категории (mcc_code) — её описание ──────────────────
# Категория = mcc_code, описание — из таблицы tr_mcc_codes
print(f"Всего категорий (уникальных mcc_code): {len(tr_mcc_codes)}")
print()
print(tr_mcc_codes[['mcc_code', 'mcc_description']].to_string(index=False))


## Вопросы к заданию 1 (4 балла)

1. Для каждой категории выведите соответствующие ей mcc-коды с описаниями.
2. Отсортируйте пользователей по количеству непустых признаков.
3. Выведите 10 id пользователей, у которых самое большое количество непустых признаков с указанием количества непустых признаков.
4. Выведите 10 id пользователей, у которых самое маленкое количество непустых признаков с указанием количества непустых признаков.

## Ответы:

**Ответы на вопросы Задания 1**

**Q1.1** — Выше выведена таблица всех mcc_code с их текстовыми описаниями. Каждый mcc_code является отдельной категорией.

**Q1.2–Q1.4** — Результаты выведены в коде выше. Клиенты отсортированы по убыванию числа ненулевых признаков (ненулевой = клиент совершал хотя бы одну транзакцию данного типа в данном mcc_code).

**Логика признакового описания**:
Приходные и расходные транзакции учитываются **раздельно**, так как они отражают разные аспекты поведения клиента. Для каждого mcc_code создаётся два признака: суммарные поступления (`_in`) и суммарные расходы (`_out`). Итоговый вектор нормализован MinMaxScaler к диапазону [0, 1]. Матрица разреженная, поскольку большинство клиентов используют лишь часть всех mcc-кодов.


# Задание 2

Модель - многослойная нейронная сеть минимум с двумя скрытыми слоями и с dropout. Количество нейронов в каждом слое выбираете самостоятельно.
Функции активации, метод оптимизации, скорость обучения, вероятность dropout - выбираете самостоятельно

Модель нужно строить с помощью keras или torch

Настроить параметры модели.

In [ ]:
# ── Задание 2: Нейронная сеть для бинарной классификации пола ─────────────

# Объединяем признаки с таргетом (только клиенты, у которых известен пол)
labeled = feature_df_norm.merge(
    gender_train.set_index('customer_id'),
    left_index=True, right_index=True,
    how='inner'
)
X_all = labeled.drop(columns=['gender']).values
y_all = labeled['gender'].values

print(f"Клиентов с известным полом: {len(X_all)}")
print(f"Признаков: {X_all.shape[1]}")
print(f"Баланс классов: 0={sum(y_all==0)}, 1={sum(y_all==1)}")

# Train / test split (80/20, стратифицированный)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Dataset
class GenderDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):   return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(GenderDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(GenderDataset(X_test,  y_test),  batch_size=64, shuffle=False)

# ── Архитектура: 2 скрытых слоя + Dropout ─────────────────────────────────
class GenderNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),   # скрытый слой 1
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),            # скрытый слой 2
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),            # скрытый слой 3
            nn.ReLU(),
            nn.Linear(16, 2),             # выходной слой (2 класса)
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_size = X_train.shape[1]
model = GenderNet(input_size).to(device)

# Подсчёт настраиваемых параметров
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nНастраиваемых параметров модели: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ── Цикл обучения (20 эпох) ────────────────────────────────────────────────
num_epochs = 20
train_losses, test_losses, train_accs, test_accs = [], [], [], []

for epoch in range(num_epochs):
    # Train
    model.train()
    rloss, correct, total = 0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        rloss += loss.item() * xb.size(0)
        _, pred = torch.max(out, 1)
        correct += (pred == yb).sum().item()
        total   += yb.size(0)
    tr_loss, tr_acc = rloss / total, correct / total

    # Test
    model.eval()
    vloss, vcorrect, vtotal = 0, 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            vloss    += criterion(out, yb).item() * xb.size(0)
            _, pred   = torch.max(out, 1)
            vcorrect += (pred == yb).sum().item()
            vtotal   += yb.size(0)
    te_loss, te_acc = vloss / vtotal, vcorrect / vtotal

    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc);   test_accs.append(te_acc)
    print(f"Epoch {epoch+1:02d}/{num_epochs} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Test  Loss: {te_loss:.4f} Acc: {te_acc:.4f}")

# Графики
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(test_losses,  label='Test Loss')
ax1.set_title('Функция потерь'); ax1.set_xlabel('Эпоха'); ax1.legend()
ax2.plot(train_accs, label='Train Accuracy')
ax2.plot(test_accs,  label='Test Accuracy')
ax2.set_title('Точность'); ax2.set_xlabel('Эпоха'); ax2.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── Улучшенная модель: фильтрация редких признаков + weight_decay + early stopping ──

# 1. Убираем редкие признаки: оставляем только те mcc-коды,
#    которые ненулевые хотя бы у 5% клиентов
threshold = 0.05 * len(feature_df_norm)
feature_df_filtered = feature_df_norm.loc[
    :, (feature_df_norm > 0).sum() >= threshold
]
print(f"Признаков после фильтрации: {feature_df_filtered.shape[1]} "
      f"(было {feature_df_norm.shape[1]})")

# Пересобираем train/test с отфильтрованными признаками
labeled2 = feature_df_filtered.merge(
    gender_train.set_index('customer_id'),
    left_index=True, right_index=True, how='inner'
)
X_all2 = labeled2.drop(columns=['gender']).values
y_all2 = labeled2['gender'].values

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_all2, y_all2, test_size=0.2, random_state=42, stratify=y_all2
)
train_loader2 = DataLoader(GenderDataset(X_train2, y_train2), batch_size=64, shuffle=True)
test_loader2  = DataLoader(GenderDataset(X_test2,  y_test2),  batch_size=64, shuffle=False)

# 2. Новая модель с Dropout(0.5)
class GenderNetV2(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.5),           # увеличили с 0.3 до 0.5
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )
    def forward(self, x): return self.net(x)

input_size2 = X_train2.shape[1]
model2 = GenderNetV2(input_size2).to(device)
total_params2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Настраиваемых параметров: {total_params2}")

# 3. Adam с weight_decay=1e-3 (L2-регуляризация)
optimizer2  = optim.Adam(model2.parameters(), lr=0.001, weight_decay=1e-3)
criterion2  = nn.CrossEntropyLoss()

# 4. Early stopping: сохраняем модель на эпохе с минимальным test loss
num_epochs2 = 20
best_test_loss  = float('inf')
best_state      = None
patience        = 5   # останавливаемся, если test loss не улучшается 5 эпох подряд
no_improve      = 0

train_losses2, test_losses2, train_accs2, test_accs2 = [], [], [], []

for epoch in range(num_epochs2):
    # Train
    model2.train()
    rloss, correct, total = 0, 0, 0
    for xb, yb in train_loader2:
        xb, yb = xb.to(device), yb.to(device)
        optimizer2.zero_grad()
        out  = model2(xb)
        loss = criterion2(out, yb)
        loss.backward()
        optimizer2.step()
        rloss   += loss.item() * xb.size(0)
        _, pred  = torch.max(out, 1)
        correct += (pred == yb).sum().item()
        total   += yb.size(0)
    tr_loss, tr_acc = rloss / total, correct / total

    # Test
    model2.eval()
    vloss, vcorrect, vtotal = 0, 0, 0
    with torch.no_grad():
        for xb, yb in test_loader2:
            xb, yb = xb.to(device), yb.to(device)
            out      = model2(xb)
            vloss   += criterion2(out, yb).item() * xb.size(0)
            _, pred   = torch.max(out, 1)
            vcorrect += (pred == yb).sum().item()
            vtotal   += yb.size(0)
    te_loss, te_acc = vloss / vtotal, vcorrect / vtotal

    train_losses2.append(tr_loss); test_losses2.append(te_loss)
    train_accs2.append(tr_acc);   test_accs2.append(te_acc)
    print(f"Epoch {epoch+1:02d}/{num_epochs2} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Test  Loss: {te_loss:.4f} Acc: {te_acc:.4f}")

    # Early stopping
    if te_loss < best_test_loss:
        best_test_loss = te_loss
        best_state     = {k: v.clone() for k, v in model2.state_dict().items()}
        no_improve     = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\nEarly stopping на эпохе {epoch+1} (best test loss: {best_test_loss:.4f})")
            break

# Загружаем лучшие веса
model2.load_state_dict(best_state)
print(f"\nЗагружены веса с минимальным test loss: {best_test_loss:.4f}")

# Графики
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses2, label='Train Loss')
ax1.plot(test_losses2,  label='Test Loss')
ax1.set_title('Функция потерь (улучшенная модель)')
ax1.set_xlabel('Эпоха'); ax1.legend()
ax2.plot(train_accs2, label='Train Accuracy')
ax2.plot(test_accs2,  label='Test Accuracy')
ax2.set_title('Точность (улучшенная модель)')
ax2.set_xlabel('Эпоха'); ax2.legend()
plt.tight_layout(); plt.show()

# Переключаем test_loader на улучшенную модель для Задания 3
model       = model2
test_loader = test_loader2


## Вопросы к заданию 2 (3 балла)

1. Перечислите все гиперпараметры и их значения вот в таком формате (значения приведены примерные, у вас могут быть другие):

* количество эпох: 5
* скорость обучения: 0.0001
* функция активация на скрытых слоях: ReLU
* и т.д.

2. Сколько у вашей модели настраиваемых в процессе обучения параметров?


## Ответы:

**Ответы на вопросы Задания 2**

**Q2.1 — Гиперпараметры улучшенной модели:**

| Гиперпараметр | Значение |
|---|---|
| Количество эпох | до 20 (с early stopping, patience=5) |
| Скорость обучения (lr) | 0.001 |
| Оптимизатор | Adam |
| L2-регуляризация (weight_decay) | 0.001 |
| Функция активации (скрытые слои) | ReLU |
| Вероятность dropout | 0.5 |
| Размер батча | 64 |
| Функция потерь | CrossEntropyLoss |
| Архитектура | input → 64 → 32 → 16 → 2 |
| Фильтрация признаков | только mcc-коды с ненулевым значением у ≥5% клиентов |
| test_size | 0.2 |
| random_state | 42 |

**Q2.2 — Количество настраиваемых параметров:**

Выведено кодом выше (`total_params2`). Зависит от числа признаков после фильтрации.


# Задание 3


Проверить качество модели не менее, чем на трёх разных метриках.

In [ ]:
# ── Задание 3: Оценка качества модели на трёх и более метриках ───────────

model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out   = model(xb)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        _, pred = torch.max(out, 1)
        all_preds.extend(pred.cpu().numpy())
        all_probs.extend(probs)
        all_labels.extend(yb.numpy())

acc       = accuracy_score(all_labels, all_preds)
f1        = f1_score(all_labels, all_preds)
roc_auc   = roc_auc_score(all_labels, all_probs)
precision = precision_score(all_labels, all_preds)
recall    = recall_score(all_labels, all_preds)

print("─── Метрики на тестовой выборке ───")
print(f"Accuracy:  {acc:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print()
print(classification_report(all_labels, all_preds,
                             target_names=['Женщина (0)', 'Мужчина (1)']))

# Важность признаков: средний вес первого слоя по абсолютной величине
feature_importance = model.net[0].weight.abs().mean(dim=0).detach().cpu().numpy()
feat_names = feature_df_filtered.columns.tolist()  # используем отфильтрованные признаки
importance_df = pd.DataFrame({'feature': feat_names, 'importance': feature_importance})
importance_df = importance_df.sort_values('importance', ascending=False)

print("─── Важность признаков (средний |вес| первого слоя) ───")
print(importance_df.to_string(index=False))


## Вопросы к заданию 3 (3 балла)

1. Выведите значения метрик.
2. Как улучшить модель?
3. Для чего может понадобиться предсказание пола клиента по его транзакциям?
4. Какие признаки оказались наиболее информативными?

## Ответы:

**Ответы на вопросы Задания 3**

**Q3.1 — Значения метрик на тестовой выборке:**

| Метрика | Значение |
|---|---|
| Accuracy | 0.6909 |
| F1-Score | 0.7463 |
| ROC-AUC | 0.7614 |
| Precision | 0.6944 |
| Recall | 0.8065 |

Модель лучше распознаёт мужчин (F1=0.75, Recall=0.81), чем женщин (F1=0.60, Recall=0.54) — вероятно, из-за дисбаланса классов в выборке (62 мужчины против 48 женщин).

**Q3.2 — Как улучшить модель:**

1. **Обогатить признаки**: добавить количество транзакций (не только суммы), среднюю сумму, максимальную, признаки по времени суток и дням недели.
2. **Балансировка классов**: применить `class_weight` в функции потерь или oversampling (SMOTE), чтобы модель лучше распознавала женщин.
3. **Снизить разреженность**: попробовать PCA или другое снижение размерности вместо фильтрации по порогу.
4. **Попробовать другие архитектуры**: градиентный бустинг (CatBoost, XGBoost) на таких признаках часто работает лучше нейросетей.
5. **Увеличить обучающую выборку**: сейчас только ~780 клиентов с известным полом — мало для нейросети.

**Q3.3 — Для чего может понадобиться предсказание пола:**

- **Персонализация предложений**: реклама и акции, ориентированные на конкретный сегмент клиентов.
- **Маркетинговая аналитика**: сравнение потребительских паттернов мужчин и женщин.
- **Сегментация клиентской базы**: уточнение профилей для кредитного скоринга.
- **Восполнение пропущенных данных**: пол известен только для части клиентов — модель позволяет заполнить пробелы.

**Q3.4 — Наиболее информативные признаки:**

По среднему абсолютному весу первого слоя нейросети (таблица выше) наиболее важными оказались расходные транзакции в категориях розничной торговли: **5812_out** (рестораны), **5977_out**, **5921_out** (продуктовые магазины), **5992_out**, **5814_out**. Это логично: паттерны расходов в продуктовых магазинах, кафе и магазинах одежды (5621, 5631) хорошо разделяют мужчин и женщин.


После завершения контрольной работы, дайте ссылку на colab в отдельном followup в своей теме на Piazza. Дайте разрешение на его просмотр.

Не забудьте запустить код в последней ячейке для контроля времени выполнения.

In [ ]:
current_datetime = datetime.now()
print(current_datetime)